# Error Handling & Recovery

*Notebook 06*

Tools fail. APIs time out. Data is missing.

An agent without error handling crashes on the first unexpected input.

An agent with error handling keeps going.


---

## 🔧 Setup

In [ ]:
import time
import random
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

Error handling is the difference between a demo and a production tool.

---

## 💥 Part 1: What Happens Without Error Handling

By default, the SDK catches tool exceptions and returns a generic error message to the model.

We disable that default here with `failure_error_function=None` so the raw exception crashes the run, making the failure mode visible.

In [ ]:
# Simulated database: only some records exist
PRODUCT_DB = {
    "P001": {"name": "Wireless Headphones", "price": 149.99, "stock": 23},
    "P002": {"name": "Phone Case", "price": 19.99, "stock": 0},
    "P003": {"name": "Laptop Stand", "price": 49.99, "stock": 7}
}


@function_tool(failure_error_function=None)  # Demo-only: surfaces the raw exception. Don't set this on real tools.
def lookup_product_fragile(product_id: str) -> str:
    """Look up product details by ID."""
    product = PRODUCT_DB[product_id]  # KeyError if product not found
    return f"{product['name']}: ${product['price']:.2f} ({product['stock']} in stock)"


fragile_agent = Agent(
    name="FragileAgent",
    instructions="Look up product information using the available tool.",
    model=MODEL,
    tools=[lookup_product_fragile]
)

# --------------------------------------------------------------
print("✅ Fragile agent ready, intentionally broken")

#### What Happens on a Bad Input

In [ ]:
print("💥 UNHANDLED EXCEPTION DEMO")
print("=" * 60)

try:

    result = await Runner.run(fragile_agent, input="What's the price of product P999?")

    print(result.final_output)

except Exception as e:
    print(f"❌ Run failed: {type(e).__name__}: {e}")
    print("   The entire agent run crashed. User gets nothing")

print("=" * 60)

---

## 🛡️ Part 2: Handling Errors Inside the Tool

By default the SDK already converts tool exceptions into a generic error message for the model.

Adding try/except inside the tool gives you control over that message.

The agent can recover with useful, actionable information instead of a vague 'tool errored' string.

In [ ]:
@function_tool
def lookup_product(product_id: str) -> str:
    """Look up product details by ID."""
    try:
        product = PRODUCT_DB[product_id]
        stock_status = "in stock" if product["stock"] > 0 else "out of stock"
        return f"{product['name']}: ${product['price']:.2f} ({product['stock']} units {stock_status})"
    except KeyError:
        return f"Product '{product_id}' not found. Available products: {', '.join(PRODUCT_DB.keys())}"
    except Exception as e:
        return f"Error looking up product: {e}"


robust_instructions = (
    "Look up product information using the available tool.\n"
    "If a product is not found, tell the user and suggest alternatives."
)

robust_agent = Agent(
    name="RobustAgent",
    instructions=robust_instructions,
    model=MODEL,
    tools=[lookup_product]
)

# --------------------------------------------------------------
print("✅ Robust agent ready")

#### Valid Product vs Missing Product

In [ ]:
print("🛡️ ROBUST TOOL DEMO")
print("=" * 60)

for query in [
    "What's the price of product P001?",
    "What's the price of product P999?"
]:
    print(f"\n🙋 {query}")

    result = await Runner.run(robust_agent, input=query)

    print(f"🤖 {result.final_output}")

print("\n" + "=" * 60)

---

## 🔁 Part 3: Retry Pattern

Some failures are transient: a flaky API, a rate limit, a momentary network issue.

For these, retry with a limit rather than failing immediately.

In [ ]:
# Simulates an unreliable external API: fails 60% of the time
def unreliable_api_call(query: str) -> str:
    """Simulates a flaky external service."""
    if random.random() < 0.6:
        raise ConnectionError("Service temporarily unavailable")
    return f"API result for: {query}"


@function_tool
def fetch_with_retry(query: str) -> str:
    """Fetch data from an external service with automatic retry."""
    max_attempts = 3
    wait_seconds = 1

    for attempt in range(1, max_attempts + 1):
        try:
            result = unreliable_api_call(query)
            print(f"    ✅ Succeeded on attempt {attempt}")
            return result
        except ConnectionError as e:
            print(f"    ⚠️  Attempt {attempt} failed: {e}")
            if attempt < max_attempts:
                time.sleep(wait_seconds)

    return f"Service unavailable after {max_attempts} attempts. Please try again later."


retry_agent = Agent(
    name="RetryAgent",
    instructions="Fetch information using the available tool. Report the result.",
    model=MODEL,
    tools=[fetch_with_retry]
)

# --------------------------------------------------------------
print("✅ Retry agent ready")

#### Run with Retry

In [ ]:
random.seed(1)  # reproducible demo: fails once, then succeeds

print("🔁 RETRY PATTERN DEMO")
print("=" * 60)
print("Fetching from unreliable service (60% failure rate)...\n")

result = await Runner.run(retry_agent, input="Fetch the latest sales report")

print(f"\n🤖 {result.final_output}")
print("=" * 60)

### 💡 Key Takeaway

Keep retries inside the tool: the agent never knows a retry happened.

⚠️ **Security note:** Only retry read-only or safe-to-repeat actions (operations that produce the same result whether run once or multiple times).

Tools that create orders, send messages, or write records may have partially succeeded. Retrying risks duplication.

---

## 💪 Practice Exercise

### Robust Calculator

*Covers: structured tool arguments and crash-safe error handling*

Build a calculator tool that handles division by zero, invalid inputs, and overflow gracefully.

In [ ]:
# --------------------------------------------------------------
# 💪 Robust Calculator
# --------------------------------------------------------------
# Objective: A calculator tool that never crashes, always returns a useful result or a clear error message.

# TODO 1: Create a @function_tool called safe_calculate
#            Parameters: expression (str), e.g. "10 / 0" or "2 ** 100"
#            Use eval() to evaluate the expression
#            Catch: ZeroDivisionError, ValueError, SyntaxError, OverflowError, NameError
#            Add a final broad except Exception fallback
#            Return a descriptive error message for each failure case
#            ⚠️  eval() is unsafe in production, fine for this exercise only

# TODO 2: Create an agent that uses safe_calculate

# TODO 3: Test with these inputs:
#            - "What is 144 / 12?"
#            - "What is 10 / 0?"
#            - "What is this: @#$%?"
#            Print each result, confirm no run crashes

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Handle errors inside the tool:**

- Use `try/except` to catch known failure modes

- Return a descriptive string, never let an exception propagate to the agent run

- Catch specific exceptions (`KeyError`, `ConnectionError`) before broad `Exception`
<br>
<br>

**Retry transient failures:**

- Set a `max_attempts` limit, never retry without a ceiling

- Add a short `time.sleep()` between attempts to avoid hammering a struggling service

- Return a clear failure message if all attempts are exhausted

- Retry only read-only or safe-to-repeat actions

---

## 📍 Next Step

**Notebook 07: Testing & Evaluating Agents**

Build a golden test set, measure what's actually working, and catch regressions early.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-06-error-handling-and-recovery)

---